In [2]:
from langgraph.graph import StateGraph, START, END
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from langchain_core.messages import BaseMessage
from langchain_ollama import OllamaLLM
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [3]:
llm = OllamaLLM(model="gemma3:4b")

In [4]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    

In [ ]:
def chat_node(state : ChatState):
    decision = interrupt({
        "type": "approval",
        "reason": "Model is about to answer a user question",
        "question": state["messages"][-1].content,
        "instructions": "Approve this question? Yes/No",
    })
    if decision["approved"] == "no":
        return {"messages": [AIMessage(content="Your question was rejected by the moderator. Please try asking something else.")]}
    else:
        response = llm.invoke(state["messages"])
        return {"messages": [AIMessage(content=response)]}

In [ ]:
builder = StateGraph(ChatState)

builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

checkpointer = MemorySaver()

app = builder.compile(checkpointer=checkpointer)


In [ ]:
config = {"configurable": {"thread_id": "1"}}

initial_input = {
    "messages": [
        {"user": "What is the capital of France?"}
    ]
}

result = app.invoke(initial_input, config = config)



In [ ]:
message = result['__interrupt__'][0].value
message

In [ ]:
user_input = input("Enter your decision (yes/no): ")


In [ ]:
final_result = app.invoke(
    Command(resume = {"approved": user_input}),
    cofig = config
)